## 1. 뉴스제목 가져오기
* user-agent 요청헤더를 반드시 설정해야 한다.

In [54]:
# requests 라이브러리 설치여부 확인
!pip show requests

Name: requests
Version: 2.31.0
Summary: Python HTTP for Humans.
Home-page: https://requests.readthedocs.io
Author: Kenneth Reitz
Author-email: me@kennethreitz.org
License: Apache 2.0
Location: C:\anaconda\Lib\site-packages
Requires: certifi, charset-normalizer, idna, urllib3
Required-by: anaconda-catalogs, anaconda-client, anaconda-cloud-auth, anaconda-project, conda, conda-build, conda-repo-cli, conda_package_streaming, cookiecutter, datashader, intake, jupyterlab_server, panel, requests-file, requests-toolbelt, Sphinx, streamlit, tldextract


In [55]:
# beautifulsoup4 라이브러리 설치여부 확인
!pip show beautifulsoup4

Name: beautifulsoup4
Version: 4.12.2
Summary: Screen-scraping library
Home-page: 
Author: 
Author-email: Leonard Richardson <leonardr@segfault.org>
License: 
Location: C:\anaconda\Lib\site-packages
Requires: soupsieve
Required-by: conda-build, nbconvert


In [56]:
# reqeusts, bs4 import
import requests
import bs4
# BeautifulSoup 클래스 import
from bs4 import BeautifulSoup

In [57]:
# requests, bs4 버전 확인하기
print(f'requests 버전 = {requests.__version__}')
print(f'bs4 버전 = {bs4.__version__}')

requests 버전 = 2.31.0
bs4 버전 = 4.12.2


### 1. 뉴스 제목 추출하기

In [58]:
# IT/과학 뉴스 
#url = 'https://news.naver.com/section/105'

# dict타입으로 요청 파라미터 설정
req_param = {
    'sid': 105
}
# 
url = 'https://news.naver.com/section/{sid}'.format(**req_param)
print(url)

# 요청 헤더 설정 : 브라우저 정보 ( 사람처럼 보이게 하기 위함 )
req_header = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

# requests 의 get() 함수 호출하기 
res = requests.get(url, headers=req_header)
print(res.status_code)
print(res.ok)
print(type(res))
#print(res.text)
# 응답(response)이 OK 이면
# 응답 (response)에서 text 추출
# BeautifulSoup 객체 생성  
if res.ok:
    soup = BeautifulSoup(res.text,'html.parser')
    print(len(soup.select("div.sa_text a[href*='https://n.news.naver.com/mnews/']")))
    # CSS 선택자를 사용해서 a tag 목록 가져오기
    a_tags = soup.select("div.sa_text a[href*='https://n.news.naver.com/mnews/']")
    print(type(a_tags), type(a_tags[0])) # [Tag,Tag]
    # <a> 태그 리스트 순회하기    
    for a_tag in a_tags:
        title = a_tag.text.strip()
        link = a_tag['href']
        print(title, link)
else:
    # 응답(response)이 Error 이면 status code 출력    
    print(f'Error Code = {res.status_code}')

https://news.naver.com/section/105
200
True
<class 'requests.models.Response'>
88
<class 'bs4.element.ResultSet'> <class 'bs4.element.Tag'>
‘K-바이오’ 창업 전초기지 오송에 문 연다 https://n.news.naver.com/mnews/article/016/0002597104
 https://n.news.naver.com/mnews/article/comment/016/0002597104
작년 농사 잘 지은 넷마블… ‘뱀피르’로 웃었지만 올해는 신작 부진 우려(종합) https://n.news.naver.com/mnews/article/366/0001140864
 https://n.news.naver.com/mnews/article/comment/366/0001140864
“AI와 친해져라”... 한컴, 전 임직원 대상 학습 조직 출범 https://n.news.naver.com/mnews/article/011/0004587446
 https://n.news.naver.com/mnews/article/comment/011/0004587446
'우주 셀카' 시대 열린다…NASA, 우주비행사 '스마트폰 지참' 첫 허용 https://n.news.naver.com/mnews/article/003/0013753599
 https://n.news.naver.com/mnews/article/comment/003/0013753599
'탈팡' 반사효과 누리더니…네이버, 커머스 앞세워 사상 최고 실적 https://n.news.naver.com/mnews/article/015/0005247833
 https://n.news.naver.com/mnews/article/comment/015/0005247833
네이버 "배송, 제약아닌 선택되도록 적극 투자…3년 내 최소 3배 향상"[컨콜] https://n.news.naver.com/mnews/article/374/00

### 1.1 뉴스제목 추출하는 함수 선언하기

In [59]:
import requests
from bs4 import BeautifulSoup

#section_dict = {100:'정치',101:'경제',102:'사회',103:'생활/문화',104:'세계',105:'IT/과학'}
section_dict = {'정치':100,'경제':101,'사회':102,'생활/문화':103,'세계':104,'IT/과학':105}

def print_news(section_name):  #print_new('생활/문화') 
    sid = section_dict.get(section_name,'정치')
    url = f'https://news.naver.com/section/{sid}'
    print(f'{section_name} 뉴스 {url}')
    req_header = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
    }
    res = requests.get(url, headers=req_header)
    if res.ok:
        soup = BeautifulSoup(res.text,'html.parser')
        # CSS 선택자를 사용해서 a tag 목록 가져오기
        a_tags = soup.select("div.sa_text a[href*='https://n.news.naver.com/mnews/']")
        # <a> 태그 리스트 순회하기    
        for a_tag in a_tags:
            title = a_tag.text.strip()
            link = a_tag['href']
            print(title, link)
    else:
        # 응답(response)이 Error 이면 status code 출력    
        print(f'Error Code = {res.status_code}')

In [60]:
print_news('경제')

경제 뉴스 https://news.naver.com/section/101
HD현대, 5800억원 규모 협력사 자재대금 조기 지급 https://n.news.naver.com/mnews/article/277/0005717521
 https://n.news.naver.com/mnews/article/comment/277/0005717521
롯데쇼핑, 올해 실적 목표 하향 조정…영업익 6500억 기존 목표 대비 18.7%↓ https://n.news.naver.com/mnews/article/421/0008757709
 https://n.news.naver.com/mnews/article/comment/421/0008757709
LS에코에너지, 사상 최대 실적 경신…2년 연속 두 자릿수 성장 https://n.news.naver.com/mnews/article/087/0001172559
 https://n.news.naver.com/mnews/article/comment/087/0001172559
‘트럼프 랠리’ 지웠다...비트코인 6만2000달러 선까지 급락 https://n.news.naver.com/mnews/article/023/0003957534
 https://n.news.naver.com/mnews/article/comment/023/0003957534
배민, 기후부와 전기 이륜차 보급 활성화 '맞손' https://n.news.naver.com/mnews/article/277/0005717586
 https://n.news.naver.com/mnews/article/comment/277/0005717586
소상공인단체 "대형마트 새벽배송 허용 추진 즉각 중단하라"(종합) https://n.news.naver.com/mnews/article/001/0015889620
 https://n.news.naver.com/mnews/article/comment/001/0015889620
다음주부터 모바일로도 로또 살 수 있다… “회차별 1인당 5000원 구매 제

### 2. Image 다운로드
* referer 요청 헤더를 반드시 설정해야 한다.

In [61]:
import requests
import os

# 육아일기 73회차
req_header = {
    'referer':'https://comic.naver.com/webtoon/detail?titleId=812354&no=208&week=sun',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

img_urls = [
    'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg',
    'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg',
    'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_3.jpg'
]

for img_url in img_urls:
    # requests 의 get(url, headers) 함수 호출하기 
    res = requests.get(img_url, headers=req_header)
    print(res.status_code)        
    # binary 응답 데이터 가져오기
    img_data = res.content    
    # url에서 파일명만 추출하기
    file_name = os.path.basename(img_url)
    print(file_name)        
    # binday data를 file에 write하기
    with open(file_name,'wb') as file:
        print(f'Writing to {file_name}({len(img_data):,} bytes)')
        file.write(img_data)

200
20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg
Writing to 20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg(124,462 bytes)
200
20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg
Writing to 20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg(141,956 bytes)
200
20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_3.jpg
Writing to 20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_3.jpg(170,913 bytes)


* 현재 요청된 페이지의 image 모두 다운로드 해보기

In [62]:
import requests
from bs4 import BeautifulSoup
import os

webtoon_url = 'https://comic.naver.com/webtoon/detail?titleId=812354&no=208&week=thu'

req_header = {
    'referer':webtoon_url,
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

res = requests.get(webtoon_url, headers=req_header)
if res.ok:
    # .jpg 파일명을 추출해서 list에 저장하기
    soup = BeautifulSoup(res.text,'html.parser')
    print(len(soup.select("img[src*='IMAG01']")))
    img_tags = soup.select("img[src*='IMAG01']")
    # img_url_list = [] #list()
    # for img_tag in img_tags:
    #     img_url = img_tag['src']
    #     img_url_list.append(img_url)

    # List Comprehension        
    img_url_list2 = [img_tag['src'] for img_tag in img_tags]    
    print(img_url_list2[:2])

    imgdir_name = 'img'
    if not os.path.isdir(imgdir_name):
        os.mkdir(imgdir_name)

    for img_url in img_url_list2:
        # requests 의 get(url, headers) 함수 호출하기 
        res = requests.get(img_url, headers=req_header)
        # binary 응답 데이터 가져오기
        img_data = res.content    

        #img/xxxIMG01.jpg
        file_path = os.path.join(imgdir_name,os.path.basename(img_url))
        # binday data를 file에 write하기
        with open(file_path,'wb') as file:
            print(f'Writing to {file_path}({len(img_data):,} bytes)')
            file.write(img_data)        
else:
    print(f'Error Code = {res.status_code}')

13
['https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg', 'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg']
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg(124,462 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg(141,956 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_3.jpg(170,913 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_4.jpg(164,871 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_5.jpg(132,142 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_6.jpg(131,414 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_7.jpg(116,752 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_8.jpg(143,277 bytes)
Writing to img\20251223155029_0d14a9e6

### 3. 파일 업로드 하기
* http://httpbin.org/post 업로드 요청을 할 수 있는 url

In [ ]:
import requests

upload_files = {
    
}
#print(upload_files)

url = 'http://httpbin.org/post'
# file 업로드 하려면 requests의 post 함수에 files 속성을 사용합니다.



200


### 4. 캡챠(이미지) API 호출하기
* urllib 사용
* 1. 캡차 키 발급 요청
  2. 캡차 이미지 요청
  3. 사용자 입력값 검증 요청

In [ ]:
# 캡차 키 발급 요청


In [ ]:
# 캡차 이미지 요청


In [ ]:
#  사용자 입력값 검증 요청



* requests를 사용하는 코드로 변경하기
* [requests docs](https://requests.readthedocs.io/en/latest/user/quickstart/)

In [ ]:
# 사용자 입력값 검증 요청



Error Code: 403


### 5. 블로그 검색하기

In [ ]:
import requests
import pprint

headers = {
    'X-Naver-Client-Id': '',
    'X-Naver-Client-Secret': '',
}

payload = {
    'query': '파이썬',
    'display': 100,
    'sort': 'sim'
}

url = 'https://openapi.naver.com/v1/search/blog.json'


# requests get(url, params, headers) 요청 

# json() 함수로 응답 결과 가져오오기
# 'title' , 'bloggername' , 'description' , 'bloggerlink' , 'link'

# 'data/nhnblog.txt' 파일 생성하기
